# 01 — Data Audit: Lending Club Accepted Loans (2007–2018Q4)

Goal: understand the raw file well enough to make defensible, documented decisions
about the target definition, the usable date range, and which columns are safe to
use — before any feature engineering happens in `02_`.

This notebook does not do EDA or modeling. It only establishes facts about the data
and records the decisions those facts force us to make.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 2. Load data

~1.6 GB, ~2.9M rows, 151 columns. Load low-cardinality string columns as `category` to keep memory manageable (mixed-type/object columns are much more expensive than the bytes on disk).

In [2]:
DATA_PATH = '../data/accepted_2007_to_2018Q4.csv'

# Low-cardinality columns: load as category instead of object to save memory.
category_cols = [
    'term', 'grade', 'sub_grade', 'emp_length', 'home_ownership',
    'verification_status', 'loan_status', 'pymnt_plan', 'purpose',
    'initial_list_status', 'application_type', 'hardship_flag',
    'debt_settlement_flag', 'disbursement_method',
]
dtype_map = {c: 'category' for c in category_cols}

df = pd.read_csv(DATA_PATH, dtype=dtype_map, low_memory=False)
df.shape


(2260701, 151)

In [3]:
df.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: category(14), float64(113), object(24)
memory usage: 4.3 GB


In [4]:
df.head()


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,"3,600.00","3,600.00","3,600.00",36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,"55,000.00",Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.00,Aug-2003,675.00,679.00,1.00,30.00,NaN,7.00,0.00,"2,765.00",29.70,13.00,w,0.00,0.00,"4,421.72","4,421.72","3,600.00",821.72,0.00,0.00,0.00,Jan-2019,122.67,NaN,Mar-2019,564.00,560.00,0.00,30.00,1.00,Individual,NaN,NaN,NaN,0.00,722.00,"144,904.00",2.00,2.00,0.00,1.00,21.00,"4,981.00",36.00,3.00,3.00,722.00,34.00,"9,300.00",3.00,1.00,4.00,4.00,"20,701.00","1,506.00",37.20,0.00,0.00,148.00,128.00,3.00,3.00,1.00,4.00,69.00,4.00,69.00,2.00,2.00,4.00,2.00,5.00,3.00,4.00,9.00,4.00,7.00,0.00,0.00,0.00,3.00,76.90,0.00,0.00,0.00,"178,050.00","7,746.00","2,400.00","13,734.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,"24,700.00","24,700.00","24,700.00",36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,"65,000.00",Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,1.00,Dec-1999,715.00,719.00,4.00,6.00,NaN,22.00,0.00,"21,470.00",19.20,38.00,w,0.00,0.00,"25,679.66","25,679.66","24,700.00",979.66,0.00,0.00,0.00,Jun-2016,926.35,NaN,Mar-2019,699.00,695.00,0.00,NaN,1.00,Individual,NaN,NaN,NaN,0.00,0.00,"204,396.00",1.00,1.00,0.00,1.00,19.00,"18,005.00",73.00,2.00,3.00,"6,472.00",29.00,"111,800.00",0.00,0.00,6.00,4.00,"9,7

## 3. Structural checks



In [5]:
df['id'].isnull().sum()


np.int64(0)

In [6]:
# Null id is 0 above, but that doesn't mean the trailer rows are gone -- check
# for ids that aren't a valid numeric loan id instead.
non_numeric_id = pd.to_numeric(df['id'], errors='coerce').isnull()
non_numeric_id.sum()


np.int64(33)

In [7]:
df.loc[non_numeric_id, ['id', 'issue_d', 'loan_status']]


,id,issue_d,loan_status
421095,Total amount funded in policy code 1: 6417608175,NaN,NaN
421096,Total amount funded in policy code 2: 1944088810,NaN,NaN
528961,Total amount funded in policy code 1: 1741781700,NaN,NaN
528962,Total amount funded in policy code 2: 564202131,NaN,NaN
651664,Total amount funded in policy code 1: 1791201400,NaN,NaN
651665,Total amount funded in policy code 2: 651669342,NaN,NaN
749520,Total amount funded in policy code 1: 1443412975,NaN,NaN
749521,Total amount funded in policy code 2: 511988838,NaN,NaN
877716,Total amount funded in policy code 1: 2063142975,NaN,NaN
877717,Total amount funded in policy code 2: 823319310,NaN,NaN


In [8]:
df = df[~non_numeric_id].copy()
df.shape


(2260668, 151)

In [9]:
df.duplicated(subset=['id']).sum()


np.int64(0)

## 4. Parse issue date and define vintage cohorts

`issue_d` is the origination month — this is the vintage axis for the whole project.
Parse it to a real datetime and derive year/quarter cohort labels used throughout
the rest of the audit (and in `02_`/`03_`).


In [10]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')

# Confirm the trailer-row cleanup in section 3 caught every unparseable date --
# a nonzero count here would silently upcast issue_year to float (NaT.year is NaN).
df['issue_d'].isnull().sum()


np.int64(0)

In [11]:
df['issue_year'] = df['issue_d'].dt.year
df['issue_quarter'] = df['issue_d'].dt.to_period('Q')

df['issue_d'].min(), df['issue_d'].max()


(Timestamp('2007-06-01 00:00:00'), Timestamp('2018-12-01 00:00:00'))

In [12]:
df.groupby('issue_year').size()


issue_year
2007       603
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134814
2014    235629
2015    421095
2016    434407
2017    443579
2018    495242
dtype: int64

In [13]:
df['issue_quarter'].value_counts().sort_index()

issue_quarter
2007Q2        24
2007Q3       190
2007Q4       389
2008Q1      1013
2008Q2       498
2008Q3       298
2008Q4       584
2009Q1       895
2009Q2      1098
2009Q3      1364
2009Q4      1924
2010Q1      2172
2010Q2      3006
2010Q3      3568
2010Q4      3791
2011Q1      4126
2011Q2      5102
2011Q3      5876
2011Q4      6617
2012Q1      8076
2012Q2     10447
2012Q3     16133
2012Q4     18711
2013Q1     22706
2013Q2     30668
2013Q3     37571
2013Q4     43869
2014Q1     47410
2014Q2     55349
2014Q3     58726
2014Q4     74144
2015Q1     84277
2015Q2     95825
2015Q3    110489
2015Q4    130504
2016Q1    133887
2016Q2     97854
2016Q3     99120
2016Q4    103546
2017Q1     96779
2017Q2    105451
2017Q3    122701
2017Q4    118648
2018Q1    107864
2018Q2    130772
2018Q3    128194
2018Q4    128412
Freq: Q-DEC, Name: count, dtype: int64

## 5. Loan status, resolution state, and target definition

First, see what's actually in the column — including the credit-policy variants and
the rare `Default` status.


In [14]:
df['loan_status'].value_counts(dropna=False)


loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

**Decision — `is_resolved` / `target`:**
- Resolved/terminal: `Fully Paid`, `Charged Off` (+ credit-policy variants, folded in via `credit_policy_flag`). `Default` (n=40) folded into `Charged Off` — too small to warrant separate handling.
- `target` = 1 for Charged Off/Default, 0 for Fully Paid; only defined where `is_resolved`.
- `Current`, `In Grace Period`, `Late (16-30/31-120 days)` are non-terminal — excluded from `target` (would leak info from loans that haven't reached an outcome yet).

In [15]:
credit_policy_variants = {
    'Does not meet the credit policy. Status:Fully Paid': 'Fully Paid',
    'Does not meet the credit policy. Status:Charged Off': 'Charged Off',
}
df['credit_policy_flag'] = df['loan_status'].isin(credit_policy_variants).astype(int)
df['loan_status_clean'] = df['loan_status'].astype(object).replace(credit_policy_variants)

resolved_statuses = ['Fully Paid', 'Charged Off', 'Default']
bad_statuses = ['Charged Off', 'Default']

df['is_resolved'] = df['loan_status_clean'].isin(resolved_statuses)
df['target'] = np.where(
    df['is_resolved'],
    df['loan_status_clean'].isin(bad_statuses).astype(int),
    np.nan,
)

df['loan_status_clean'].value_counts(dropna=False)

loan_status_clean
Fully Paid            1078739
Current                878317
Charged Off            269320
Late (31-120 days)      21467
In Grace Period          8436
Late (16-30 days)        4349
Default                    40
Name: count, dtype: int64

In [16]:
# credit_policy_flag should be concentrated in the earliest vintages -- confirm.
df.groupby('issue_year')['credit_policy_flag'].mean()


issue_year
2007   0.58
2008   0.35
2009   0.11
2010   0.08
2011   0.00
2012   0.00
2013   0.00
2014   0.00
2015   0.00
2016   0.00
2017   0.00
2018   0.00
Name: credit_policy_flag, dtype: float64

**Censoring check:** `is_censored` = `~is_resolved`, by quarter — shows how much of each recent cohort hasn't reached an outcome yet (same effect as the term decision above).

In [17]:
df['is_censored'] = ~df['is_resolved']
df.groupby('issue_quarter')['is_censored'].mean()

issue_quarter
2007Q2   0.00
2007Q3   0.00
2007Q4   0.00
2008Q1   0.00
2008Q2   0.00
2008Q3   0.00
2008Q4   0.00
2009Q1   0.00
2009Q2   0.00
2009Q3   0.00
2009Q4   0.00
2010Q1   0.00
2010Q2   0.00
2010Q3   0.00
2010Q4   0.00
2011Q1   0.00
2011Q2   0.00
2011Q3   0.00
2011Q4   0.00
2012Q1   0.00
2012Q2   0.00
2012Q3   0.00
2012Q4   0.00
2013Q1   0.00
2013Q2   0.00
2013Q3   0.00
2013Q4   0.00
2014Q1   0.01
2014Q2   0.05
2014Q3   0.06
2014Q4   0.08
2015Q1   0.09
2015Q2   0.10
2015Q3   0.11
2015Q4   0.12
2016Q1   0.16
2016Q2   0.36
2016Q3   0.39
2016Q4   0.45
2017Q1   0.52
2017Q2   0.58
2017Q3   0.64
2017Q4   0.71
2018Q1   0.79
2018Q2   0.86
2018Q3   0.92
2018Q4   0.96
Freq: Q-DEC, Name: is_censored, dtype: float64

## 6. Null-rate audit by column, cross-tabbed against vintage

First, overall null rate per column.


In [18]:
# Show raw counts alongside the percentage -- a rate that rounds to 0.00%
# can still hide real nulls (e.g. earliest_cr_line's 29, found in section 9b)
# if only the rounded percentage is shown.
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': df.isnull().mean(),
}).sort_values('null_pct', ascending=False)
null_summary[null_summary['null_count'] > 0]

,null_count,null_pct
member_id,2260668,1.00
orig_projected_additional_accrued_interest,2252017,1.00
hardship_end_date,2249751,1.00
hardship_loan_status,2249751,1.00
hardship_last_payment_amount,2249751,1.00
hardship_reason,2249751,1.00
hardship_type,2249751,1.00
hardship_status,2249751,1.00
payment_plan_start_date,2249751,1.00
hardship_amount,2249751,1.00


Check a few high-null columns against `issue_year` — schema growth (100% null before the field existed) looks very different from structural sparsity (null throughout).

In [19]:
for col in ['mo_sin_old_rev_tl_op', 'bc_util', 'annual_inc_joint']:
    print(col)
    print(df.groupby('issue_year')[col].apply(lambda x: x.isnull().mean()))
    print()


mo_sin_old_rev_tl_op
issue_year
2007   1.00
2008   1.00
2009   1.00
2010   1.00
2011   1.00
2012   0.52
2013   0.00
2014   0.00
2015   0.00
2016   0.00
2017   0.00
2018   0.00
Name: mo_sin_old_rev_tl_op, dtype: float64

bc_util


issue_year
2007   1.00
2008   1.00
2009   1.00
2010   1.00
2011   1.00
2012   0.15
2013   0.01
2014   0.01
2015   0.01
2016   0.01
2017   0.01
2018   0.01
Name: bc_util, dtype: float64

annual_inc_joint
issue_year
2007   1.00
2008   1.00
2009   1.00
2010   1.00
2011   1.00
2012   1.00
2013   1.00
2014   1.00
2015   1.00
2016   0.98
2017   0.90
2018   0.86
Name: annual_inc_joint, dtype: float64



**Decision:** null-everywhere columns (joint/hardship/settlement/sec_app fields) are structurally sparse, not missing — drop or defer for Phase 1. Null-early-only columns (bureau feature cluster added ~2013) are usable only for `issue_year >= 2013`. Full lists in section 10.

## 7. Term distribution and cohort/term interaction


In [20]:
df['term'].value_counts()


term
36 months    1609754
60 months     650914
Name: count, dtype: int64

In [21]:
df.groupby(['issue_quarter', 'term'], observed=True).size().unstack()

term,36 months,60 months
issue_quarter,,
2007Q2,24.00,NaN
2007Q3,190.00,NaN
2007Q4,389.00,NaN
2008Q1,"1,013.00",NaN
2008Q2,498.00,NaN
2008Q3,298.00,NaN
2008Q4,584.00,NaN
2009Q1,895.00,NaN
2009Q2,"1,098.00",NaN


**Decision:** keep both terms; always segment/condition on `term` in vintage curves rather than pooling. 60-month loans take longer to resolve, so pooling would confound term with vintage — recent 60-month cohorts should read as right-censored, not low-risk.

## 8. Class balance / baseline default rate

Restricting to resolved loans (per the `is_resolved` decision in section 5) gives
the baseline bad rate — important context for interpreting AUC/KS later, and for
whether Phase 1 modeling needs class weighting or resampling.


In [22]:
resolved = df[df['is_resolved']]
resolved['target'].value_counts(normalize=True)


target
0.00   0.80
1.00   0.20
Name: proportion, dtype: float64

In [23]:
resolved.groupby('issue_year')['target'].mean()


issue_year
2007   0.26
2008   0.21
2009   0.14
2010   0.14
2011   0.15
2012   0.16
2013   0.16
2014   0.18
2015   0.20
2016   0.23
2017   0.23
2018   0.16
Name: target, dtype: float64

## 9. Sanity checks on key numeric fields

Look for placeholder/garbage values before trusting anything downstream.


In [24]:
df[['annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'revol_util']].describe()


,annual_inc,dti,fico_range_low,fico_range_high,revol_util
count,"2,260,664.00","2,258,957.00","2,260,668.00","2,260,668.00","2,258,866.00"
mean,"77,992.43",18.82,698.59,702.59,50.34
std,"112,696.20",14.18,33.01,33.01,24.71
min,0.00,-1.00,610.00,614.00,0.00
25%,"46,000.00",11.89,675.00,679.00,31.50
50%,"65,000.00",17.84,690.00,694.00,50.30
75%,"93,000.00",24.49,715.00,719.00,69.40
max,"110,000,000.00",999.00,845.00,850.00,892.30


In [25]:
# revol_util should be a percentage -- flag anything outside [0, 100].
(df['revol_util'] < 0).sum(), (df['revol_util'] > 100).sum()


(np.int64(0), np.int64(7343))

In [26]:
# dti outliers: self-employed / small-business borrowers can have extreme values.
df['dti'].sort_values(ascending=False).head(10)


452427    999.00
1413979   999.00
606144    999.00
552842    999.00
1443180   999.00
1719462   999.00
2227387   999.00
754471    999.00
683735    999.00
477824    999.00
Name: dti, dtype: float64

In [27]:
# dti also has a negative floor per the describe() above (min -1.00) -- check the low end too.
df['dti'].sort_values().head(10)


1681348   -1.00
1014615   -1.00
773884     0.00
866244     0.00
1450253    0.00
1560963    0.00
2145577    0.00
2039433    0.00
1629961    0.00
485160     0.00
Name: dti, dtype: float64

In [28]:
# dti shouldn't be negative -- count how many rows hit this.
(df['dti'] < 0).sum()


np.int64(2)

In [29]:
# annual_inc is self-reported and unverified for a meaningful share of borrowers --
# check verification_status alongside the largest incomes.
df.loc[df['annual_inc'].sort_values(ascending=False).index[:10],
       ['annual_inc', 'verification_status', 'issue_year']]


,annual_inc,verification_status,issue_year
601128,"110,000,000.00",Verified,2017
1673140,"61,000,000.00",Source Verified,2017
539807,"10,999,200.00",Source Verified,2017
1597151,"9,930,475.00",Source Verified,2018
1413812,"9,757,200.00",Source Verified,2018
735426,"9,573,072.00",Source Verified,2016
1004912,"9,550,000.00",Source Verified,2016
1684835,"9,522,972.00",Source Verified,2017
230972,"9,500,000.00",Source Verified,2015
1654748,"9,300,086.00",Not Verified,2017


### 9b. Additional sanity checks (annual_inc tail, emp_length nulls, earliest_cr_line)

`dti < 0` is already covered above (n=2, both exactly -1.00) -- folded into
`GarbageValueCleaner`'s existing `dti == 999` placeholder rule
(`src/preprocessing.py`) as a second, distinct garbage pattern (a physically
impossible ratio rather than a sentinel value), not re-derived here. The
three checks below are new -- audit only, no filters added yet.

In [30]:
# annual_inc upper tail: percentile view plus round-number thresholds,
# extending the top-10 check above with verification_status context.
print(df['annual_inc'].quantile([0.99, 0.999, 0.9999, 0.99999]))
print()
for cutoff in [1_000_000, 5_000_000, 10_000_000]:
    print(f'annual_inc > ${cutoff:,}: {(df["annual_inc"] > cutoff).sum()}')
print()
df.loc[df['annual_inc'] > 1_000_000, 'verification_status'].value_counts(dropna=False)

0.99     270,000.00
1.00     600,000.00
1.00   1,950,000.00
1.00   8,378,882.00
Name: annual_inc, dtype: float64

annual_inc > $1,000,000: 583
annual_inc > $5,000,000: 93
annual_inc > $10,000,000: 3



verification_status
Source Verified    423
Verified           130
Not Verified        30
Name: count, dtype: int64

In [31]:
# emp_length null rate, and whether null looks like a distinct population
# (e.g. unemployed/self-employed) rather than random missingness -- compare
# against the '< 1 year' bucket on the target and a couple of covariates.
print(f"emp_length null rate: {df['emp_length'].isnull().mean():.2%}")

resolved_emp = resolved.copy()
resolved_emp['emp_length_group'] = resolved_emp['emp_length'].astype(object)
resolved_emp['emp_length_group'] = resolved_emp['emp_length_group'].where(
    resolved_emp['emp_length_group'].notnull(), 'Null'
)
compare = resolved_emp[resolved_emp['emp_length_group'].isin(['Null', '< 1 year'])]

print()
print('Default rate by group (resolved loans):')
print(compare.groupby('emp_length_group', observed=True)['target'].agg(['mean', 'count']))

print()
print('Median annual_inc by group (resolved loans):')
print(compare.groupby('emp_length_group', observed=True)['annual_inc'].median())

print()
print('home_ownership mix by group (resolved loans):')
print(compare.groupby('emp_length_group', observed=True)['home_ownership'].value_counts(normalize=True).unstack())

emp_length null rate: 6.50%



Default rate by group (resolved loans):
                  mean   count
emp_length_group              
< 1 year          0.21  108537
Null              0.27   78550

Median annual_inc by group (resolved loans):
emp_length_group
< 1 year   60,000.00
Null       44,000.00
Name: annual_inc, dtype: float64

home_ownership mix by group (resolved loans):
home_ownership    ANY  MORTGAGE  NONE  OTHER  OWN  RENT
emp_length_group                                       
< 1 year         0.00      0.37  0.00   0.00 0.09  0.54
Null             0.00      0.46  0.00   0.00 0.20  0.34


In [32]:
# earliest_cr_line sanity: parsed locally, not assigned back onto df -- this
# is audit-only, and nothing downstream in this notebook needs it. Looking
# for values that would produce a negative or implausibly large
# credit_history_years when notebook 03 derives it as (issue_d - earliest_cr_line).
earliest_cr_line_parsed = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

print('earliest_cr_line parsed range:', earliest_cr_line_parsed.min(), 'to', earliest_cr_line_parsed.max())
print('Unparseable (NaT) count:', earliest_cr_line_parsed.isnull().sum())

after_issue = earliest_cr_line_parsed > df['issue_d']
print(f'\nearliest_cr_line after issue_d (negative credit history): {after_issue.sum()}')

print('\nOldest 10 earliest_cr_line values:')
print(earliest_cr_line_parsed.sort_values().head(10))

credit_history_years_check = (df['issue_d'] - earliest_cr_line_parsed).dt.days / 365.25
print('\ncredit_history_years (derived here for audit only) describe():')
print(credit_history_years_check.describe())

earliest_cr_line parsed range: 1933-03-01 00:00:00 to 2015-11-01 00:00:00
Unparseable (NaT) count: 29

earliest_cr_line after issue_d (negative credit history): 0

Oldest 10 earliest_cr_line values:


653398    1933-03-01
920381    1934-02-01
624681    1934-04-01
625148    1934-04-01
1946154   1941-08-01
1307846   1944-01-01
411227    1944-01-01
1058646   1945-02-01
1625997   1946-01-01
1146182   1946-08-01
Name: earliest_cr_line, dtype: datetime64[ns]

credit_history_years (derived here for audit only) describe():


count   2,260,639.00
mean           16.39
std             7.68
min             0.50
25%            11.25
50%            14.83
75%            20.25
max            83.25
dtype: float64


**Decision — `GarbageValueCleaner` rules (full list, `src/preprocessing.py`):**
- `dti == 999`: LendingClub's placeholder sentinel (confirmed via the sorted
  top-10 above; n=38 in notebook 03's 2013+ modeling population).
- `dti < 0`: physically impossible ratio (n=2 here, both exactly -1.00).
- `revol_util > 100`: utilization can't exceed 100% (n=7,343 here).
- `annual_inc <= 0`: income can't be zero or negative (n=361 in notebook
  03's 2013+ modeling population).
- `annual_inc > $50,000,000`: two isolated outlier rows ($110M, $61M) -- 10x+
  beyond the next-highest value (~$11M) and lacking any corroborating signal
  (both show a small `loan_amnt` and near-zero `dti`, an otherwise ordinary
  borrower profile). The $10-11M tier is left alone: same small-loan/low-dti
  pattern, but not magnitude-isolated the way the top two are, and plausibly
  real high earners.

All five rules apply identically at training and Phase 3 inference time.

**No filter added:**
- `emp_length` nulls (6.50%): a distinctly higher-risk group (27% vs. 21%
  default rate for `< 1 year`, resolved loans) -- not random missingness,
  but `SimpleImputer(strategy='median', add_indicator=True)` already exposes
  it via a missingness indicator rather than silently blending it into the
  median. Verified directly in notebook 04's SHAP section.
- `earliest_cr_line` nulls (29 rows -- hidden by section 6's rounding until
  fixed above): propagate to NaN in `credit_history_years` and go through
  the same median+indicator imputation as any other numeric feature.

## 10. Data audit summary

- **Structural cleanup:** 33 trailer/footer rows dropped (non-numeric `id`, e.g. `"Total amount funded in policy code 1: ..."`, every other column NaN) — a null-`id` check alone misses these since `id` holds text, not null. Left in, they'd upcast `issue_year` to float via `NaT`. Modeling universe is 2,260,668 rows going forward.
- **Target:** `target`=1 for Charged Off/Default (+ policy variants), 0 for Fully Paid; only defined where `is_resolved`. ~59.6% of 2,260,668 rows resolved.
- **Date range:** 2007-06 to 2018-12. Volume negligible before 2013 (<10% of rows); heavily back-loaded toward 2015-2018.
- **Term:** 36mo (1.61M) and 60mo (0.65M) both kept, segmented by `term` in vintage work. 60mo starts 2010Q2.
- **Censoring:** `is_censored` rises sharply post-2013 (~0% → 96% by 2018Q4) — recent-vintage default rates aren't comparable to seasoned vintages without accounting for this.
- **Credit-policy flag:** 2,749 loans, 100% concentrated in 2007-2010 (0% from 2011+). Kept as covariate; expected negligible effect once modeling on 2013+ data.
- **Drop (leakage/unusable):** `member_id`, `url`; all post-origination fields — `total_pymnt*`, `recoveries`, `collection_recovery_fee`, `last_pymnt_*`, `next_pymnt_d`, `last_credit_pull_d`, `last_fico_range_*`, `out_prncp*`, `hardship_*`, `debt_settlement_flag*`, `settlement_*`.
- **Defer (94%+ null, small subpopulation):** `sec_app_*`, joint-application fields (`annual_inc_joint`, `dti_joint`, `verification_status_joint`, `revol_bal_joint`), `desc`.
- **Vintage-dependent nulls:** bureau feature cluster (`mo_sin_*`, `bc_util`, `num_*`, `tot_*`, `open_*_6m/12m/24m`, etc.) ~100% null pre-2012, ~0-1.5% null from 2013+. Restrict to `issue_year >= 2013` if using these.
- **Baseline default rate:** 19.98% overall; by year ~14% (2009-10) → ~23% (2016-17) → 15.8% (2018, likely a censoring artifact, not real improvement).
- **Product framing note:** `grade`, `sub_grade`, `int_rate` are present and clean here, but are flagged for exclusion from Phase 1 modeling — Phase 1 scores a not-yet-underwritten applicant, and these fields are LendingClub's own post-underwriting output. Data-quality facts only; full rationale in notebook `02_`/`03_`.
